# 05 · Memory & context

**Course:** Building LLM Agents, notebook 5 of 10
**Where we are:** in notebook 01 we learned the model is stateless. What looks like "memory" is really just the message list *we* resend on every call. In notebook 04 we built an agent that keeps such a list as it loops. Now we manage that list deliberately, so an agent can hold long conversations and recall facts across them.

### What you'll learn
- The context window: a finite token budget (it varies by model, often 100k+ tokens), and why long chats overflow it
- Short-term memory, which is the conversation buffer, plus a sliding window to stay under budget
- Summarisation memory: use the LLM *itself* to compress old turns into a running summary
- Long-term / retrieval memory (RAG-lite): write facts to a store and fetch the relevant ones later, using lexical search (no embeddings needed)
- Where short-term and long-term memory each fit in an agent

### Prerequisites
- Notebooks 01 to 04: chat completions, statelessness and multi-turn history, tokens/usage, and the agent loop

> Mental model: the model has no memory of its own. Memory in an agent is an engineering choice about what you put back into the message list before each call. Everything in this notebook is a different answer to one question: given a finite token budget, which words do I show the model right now?

## Setup

Same one line as every notebook. We also pull in `ask` (a one-shot text call) and `show_messages` (renders a conversation). Both come straight from `helpers.py`.

In [ ]:
from helpers import client, MODEL, ask, jprint, show_messages

MODEL

## 1. The context window: a finite budget

Recall from notebook 01 that to hold a conversation you resend the entire history every turn. That history is measured in tokens, and the model can only accept so many at once: its context window. Different models allow different amounts, often anywhere from tens of thousands to a few hundred thousand tokens (prompt plus completion combined).

That sounds enormous, but an agent that loops, resending a growing transcript plus tool results on every step, can chew through it surprisingly fast. There is a second cost too: you pay (in tokens, and on a shared cluster in *time*) for every token you resend, every turn.

> The overflow problem: when the history grows past the window, you can't just keep appending. You must decide what to drop or compress, and that decision is memory management.

Let's set the budget as a constant and watch the prompt grow.

![Tokens climb each turn and eventually hit the context-window ceiling](figures/05-context-window.svg)

*Figure: history is resent every turn, so tokens climb until they hit the window ceiling. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
# The context window depends on the model your project uses; check its docs.
# We use a representative round number here just to make the demo concrete.
CONTEXT_WINDOW = 256_000  # tokens the model can hold at once (prompt + completion)
print(f"Context window: {CONTEXT_WINDOW:,} tokens")

### Watch the prompt grow

We'll have a short back-and-forth, appending each turn to one list (exactly the manual pattern from notebook 01). After every model call we read `response.usage.prompt_tokens`, the size of what we *sent in*. Because we keep resending the whole history, that number only ever climbs.

In [ ]:
messages = [{"role": "system", "content": "You are a concise assistant. Answer in one short sentence."}]

turns = [
    "Name a planet in our solar system.",
    "Name a different one.",
    "And one more, different again.",
    "Now list all three you named, in order.",
]

for user_turn in turns:
    messages.append({"role": "user", "content": user_turn})
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0, max_tokens=8000)
    reply = r.choices[0].message.content.strip()
    messages.append({"role": "assistant", "content": reply})
    u = r.usage
    print(f"prompt_tokens={u.prompt_tokens:>4}  | Q: {user_turn[:32]:32}  -> {reply}")

`prompt_tokens` rises with every turn. That's the cost of statelessness made visible. On a four-line chat it's nothing. Scale it to a 200-turn support conversation, or an agent that loops with verbose tool outputs, and you'll hit the wall. The rest of this notebook is three techniques for staying under it.

## 2. Short-term memory = the conversation buffer

The simplest memory is just the list of messages for the current conversation: the *buffer*. We've been managing it by hand. Let's wrap it in a tiny class so the intent is explicit, and give it a sliding window: a way to return only the last N turns, so the prompt can't grow without bound.

> Key idea: a *turn* here is one user/assistant exchange. Keeping the last N turns is the cheapest possible memory policy: forget the oldest stuff. It's crude, but for many assistants the recent context is all that matters.

In [ ]:
class ConversationMemory:
    """Stores a conversation and can return a trimmed view of it.

    - The system message (if any) is pinned: it never gets windowed out.
    - `window(n)` returns [system] + the last n*2 non-system messages
      (n turns = n user + n assistant messages).
    """

    def __init__(self, system=None):
        self.system = system
        self.turns: list[dict] = []  # everything except the system message

    def add(self, role: str, content: str):
        self.turns.append({"role": role, "content": content})

    def all(self) -> list[dict]:
        head = [{"role": "system", "content": self.system}] if self.system else []
        return head + self.turns

    def window(self, n: int) -> list[dict]:
        """Last n turns only (a turn = a user msg + an assistant msg)."""
        head = [{"role": "system", "content": self.system}] if self.system else []
        return head + self.turns[-2 * n:]

Let's load it with a few turns and compare the full history against a 2-turn window.

In [ ]:
mem = ConversationMemory(system="You are a travel assistant.")
mem.add("user", "I'm planning a trip to Bergen.")
mem.add("assistant", "Great — Bergen is lovely. How many days?")
mem.add("user", "Four days in July.")
mem.add("assistant", "Perfect for the fjords and the Fløibanen funicular.")
mem.add("user", "Any rain gear advice?")
mem.add("assistant", "Always pack a waterproof jacket — Bergen is Norway's rainiest city.")

print(f"FULL history : {len(mem.all())} messages")
print(f"window(n=2)  : {len(mem.window(2))} messages\n")
show_messages(mem.window(2))

The window keeps the pinned system message plus the most recent 2 turns, and silently drops the Bergen-trip opener. That bounds the prompt size, but notice what we lost: the number of days and the month are no longer in view. A pure window is forgetful by design.

> Gotcha: sliding windows throw away information the model may still need ("how many days was the trip again?"). The fix isn't a bigger window. It's to compress the old turns instead of deleting them. That's next.

![Summarisation memory: compress old turns, keep a running summary](figures/05-summarisation-loop.svg)

*Figure: old turns go to an LLM summary, you keep the summary plus recent turns, then repeat. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 3. Summarisation memory (the key technique)

Instead of *dropping* old turns, compress them. When the history gets long, hand the oldest part to the LLM and ask for a short running summary, then keep only `[system, summary, ...recent turns verbatim]`.

This is the workhorse of long-lived chat assistants: recent turns stay sharp and literal, while everything older collapses into a few sentences that still carry the key facts.

> Key idea: the model is also your compressor. You already pay to read the old turns once; a summary lets you keep their *content* for a fraction of the tokens on every future call.

First, the compressor: a plain function built on `ask()`.

In [ ]:
def summarise(old_messages: list[dict], prev_summary: str = "") -> str:
    """Compress old conversation turns into a short running summary."""
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in old_messages)
    system = (
        "You compress conversation history. Produce a terse, factual summary that "
        "preserves names, numbers, decisions, and open questions. Use 3-5 short bullet points. "
        "No preamble."
    )
    prompt = ""
    if prev_summary:
        prompt += f"Summary so far:\n{prev_summary}\n\n"
    prompt += f"New turns to fold in:\n{transcript}\n\nUpdated summary:"
    return ask(prompt, system=system, temperature=0, max_tokens=8000).strip()

Now a memory that uses it. We keep a buffer like before, but when it grows past a threshold we summarise the oldest turns and drop them, retaining only the most recent ones verbatim. The prompt we send is always `[system, summary, ...recent]`.

In [ ]:
class SummarisingMemory:
    """Keeps recent turns verbatim; folds older turns into a running summary."""

    def __init__(self, system: str, keep_recent: int = 2, summarise_after: int = 6):
        self.system = system
        self.keep_recent = keep_recent          # turns kept verbatim
        self.summarise_after = summarise_after  # compress once we exceed this many messages
        self.summary = ""
        self.turns: list[dict] = []

    def add(self, role: str, content: str):
        self.turns.append({"role": role, "content": content})
        self._maybe_compress()

    def _maybe_compress(self):
        if len(self.turns) <= self.summarise_after:
            return
        keep = 2 * self.keep_recent
        old, recent = self.turns[:-keep], self.turns[-keep:]
        self.summary = summarise(old, self.summary)
        self.turns = recent  # the old turns now live *only* inside the summary

    def messages(self) -> list[dict]:
        # The endpoint accepts exactly ONE system message, so the running
        # summary is folded into it rather than sent as a second one.
        system = self.system
        if self.summary:
            system += f"\n\nSummary of earlier conversation:\n{self.summary}"
        return [{"role": "system", "content": system}] + self.turns

Let's drive it past the threshold and watch it compress. We feed in a chat with a couple of facts buried early (the user's name and their dog's name), keep going until summarisation kicks in, then inspect what's left.

In [ ]:
chat = SummarisingMemory(system="You are a friendly assistant.", keep_recent=2, summarise_after=6)

script = [
    ("user", "Hi! My name is Mara and I live in Tromsø."),
    ("assistant", "Hi Mara! Tromsø is beautiful this time of year."),
    ("user", "I have a husky called Balto."),
    ("assistant", "Balto is a wonderful name for a husky!"),
    ("user", "I'm learning to bake sourdough."),
    ("assistant", "Sourdough takes patience — keep that starter fed."),
    ("user", "I also play the cello."),
    ("assistant", "Lovely — music and baking pair nicely."),
]
for role, content in script:
    chat.add(role, content)

print(f"Verbatim turns kept : {len(chat.turns)}  (of {len(script)} added)")
print(f"\nRunning summary:\n{chat.summary}")
print("\n--- what we'd actually send to the model ---")
show_messages(chat.messages())

The earliest turns are gone from the verbatim buffer, but their content survives in the summary system message. Now the payoff: ask about a fact from a *summarised* turn. The model can only answer if compression preserved it.

In [ ]:
chat.add("user", "What's my dog's name, and what city do I live in?")
r = client.chat.completions.create(model=MODEL, messages=chat.messages(), temperature=0, max_tokens=8000)
print(r.choices[0].message.content.strip())

It recalls Balto and Tromsø even though those turns were summarised away, because the summary kept the facts. That's the whole trick.

> Rule of thumb: keep the last few turns verbatim (the model needs literal recent context to stay coherent) and summarise the rest. Tune `keep_recent` and `summarise_after` to your conversation length and budget.

> Cost: summarising costs an extra LLM call each time it fires, but it caps the per-turn prompt size for the entire rest of the conversation. For long chats that trade pays for itself many times over.

## 4. Long-term / retrieval memory (RAG-lite)

Short-term memory lives and dies with one conversation. Long-term memory persists *across* conversations: the agent writes facts to a store, and later retrieves the relevant ones to inject into the prompt. This is the "R" and "A" of Retrieval-Augmented Generation: retrieve, then augment the prompt, then generate.

The crucial step is retrieval: given a question, find the few stored facts worth showing the model. Production systems do this with vector embeddings (semantic similarity). But:

> Our endpoint has no embeddings. `client.embeddings.create(...)` returns an error here. So we implement retrieval the dependency-free way, with lexical scoring: rank facts by how many query words they share. It's crude but genuinely useful, and it makes the mechanism obvious.

> The upgrade: in production you'd swap the scorer for an embedding model plus a vector database (FAISS, Chroma, pgvector, and so on) so "canine" can match "dog". The shape stays exactly the same: `add()` / `search()`, then inject top-k into the prompt.

Here's the store. Retrieval is just token-overlap with a small bonus for substring matches.

In [ ]:
import re

def _tokens(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))

class FactStore:
    """A tiny long-term memory: store text facts, retrieve by keyword overlap."""

    def __init__(self):
        self.facts: list[str] = []

    def add(self, text: str):
        self.facts.append(text.strip())

    def _score(self, query_tokens: set[str], fact: str) -> float:
        fact_tokens = _tokens(fact)
        overlap = len(query_tokens & fact_tokens)          # shared words
        substring = 0.5 if any(t in fact.lower() for t in query_tokens if len(t) > 3) else 0.0
        return overlap + substring

    def search(self, query: str, k: int = 3) -> list[str]:
        """Return the top-k facts most lexically relevant to the query."""
        q = _tokens(query)
        scored = [(self._score(q, f), f) for f in self.facts]
        scored = [(s, f) for s, f in scored if s > 0]      # drop zero-overlap facts
        scored.sort(key=lambda sf: sf[0], reverse=True)
        return [f for _, f in scored[:k]]

Store a handful of facts (imagine these were saved across many past sessions), then retrieve for a question. Note how only the *relevant* facts come back, not the whole store.

In [ ]:
store = FactStore()
store.add("The user's name is Mara and she lives in Tromsø.")
store.add("Mara has a husky called Balto.")
store.add("Mara is allergic to peanuts.")
store.add("The user's preferred programming language is Python.")
store.add("Mara's favourite season is winter.")

for h in store.search("What is Mara allergic to?", k=2):
    print("•", h)

The peanut-allergy fact ranks top: it shares the words *Mara* and *allergic* with the query, so it scores highest. Now the full RAG-lite loop: retrieve relevant facts, drop them into the prompt as context, and let the model answer grounded in them.

In [ ]:
def answer_with_memory(question: str, store: FactStore, k: int = 3) -> str:
    facts = store.search(question, k=k)
    context = "\n".join(f"- {f}" for f in facts) or "(no relevant facts found)"
    system = (
        "Answer using ONLY the facts provided. "
        "If the facts don't cover it, say you don't know."
    )
    prompt = f"Known facts:\n{context}\n\nQuestion: {question}"
    print("Retrieved facts:")
    for f in facts:
        print("  •", f)
    print()
    return ask(prompt, system=system, temperature=0, max_tokens=8000).strip()

print(answer_with_memory("What is Mara allergic to?", store))

The agent answered from a fact it stored in a *previous* session. That's long-term memory in action. The model was never told about the allergy in this conversation; retrieval put it in front of the model exactly when it became relevant.

> Rule of thumb: don't dump the whole store into the prompt, because that just recreates the context-window problem. Retrieve top-k and inject only those. Retrieval is how you give an agent access to far more knowledge than fits in its window.

### Where lexical search breaks (and why embeddings exist)

Lexical scoring only matches shared words. Phrase the same question with a synonym and it falls apart. Watch the *exact same* allergy fact vanish from the results when we ask about "allergies" instead of "allergic":

In [ ]:
print('query: "What is Mara allergic to?"')
for h in store.search("What is Mara allergic to?", k=2):
    print("   •", h)

print('\nquery: "Does Mara have any allergies?"   <- same meaning, different words')
for h in store.search("Does Mara have any allergies?", k=2):
    print("   •", h)

The second query never surfaces the peanut fact: "allergies" simply isn't the token "allergic", so the score is zero. A human sees these as the same question; the lexical scorer does not.

> The upgrade, embeddings: production retrieval encodes each fact and the query as a vector and ranks by *semantic* similarity, so "allergies" matches "allergic" and "canine" matches "husky", no shared words required. Our endpoint doesn't expose embeddings, so we used lexical scoring as the dependency-free stand-in. Swap `_score` for a vector similarity against an embedding index and everything else here is unchanged.

## 5. Tying it together: short-term vs long-term

Two different jobs, two different mechanisms:

| | **Short-term memory** | **Long-term memory** |
|---|---|---|
| Scope | *this* conversation | *across* conversations / sessions |
| Built from | the message buffer | an external store (DB, vector index, file) |
| Stays small via | windowing + summarisation | retrieval of top-k relevant items |
| Answers | "what did we just say?" | "what do I know about this user/topic?" |
| In this notebook | §2 buffer, §3 summary | §4 `FactStore` |

A real agent uses both at once: it keeps a windowed/summarised buffer of the live conversation and retrieves relevant long-term facts to inject, both feeding into the one message list it sends each turn.

> Why you care: from notebook 06 on, our agents loop and call tools, and that transcript grows fast. The memory policy you choose (window, summarise, retrieve, or a blend) is what keeps an agent affordable, coherent, and able to remember the user from yesterday.

## Recap

You can now:
- explain the context window (a finite, model-dependent token budget) and watch `prompt_tokens` climb as history grows
- keep a conversation buffer and bound it with a sliding window (`ConversationMemory`)
- use the LLM to summarise old turns into a running summary, keeping recent turns verbatim (`SummarisingMemory`), the key technique
- build long-term retrieval memory with `FactStore`: `add()` facts, `search()` the relevant top-k by lexical scoring, inject them into the prompt (RAG-lite)
- distinguish short-term (this conversation) from long-term (across conversations) memory and place each in an agent

### Try it yourself
1. Lower `summarise_after` to `4` in §3 and re-run. How does the summary change, and what does it cost in extra calls?
2. Add `"Mara is vegetarian."` to the `FactStore`, then ask `"What is Mara allergic to and does she eat meat?"`. Do both diet facts get retrieved? Now ask `"What can the user eat?"` and see why retrieval does worse here.
3. Make lexical search a little smarter: extend `_score` so the query word `"dog"` also matches the fact about a `"husky"` (hint: a tiny hand-written synonym map). This is the poor-man's version of what embeddings do automatically.
4. Give `ConversationMemory` a `total_tokens()` method (use `len(text.split())` as a rough proxy) and trigger windowing by a token budget instead of a turn count.

### Next up
**06 · A useful single agent.** We take the `Agent` from notebook 04, give it a real multi-tool toolbox, add error handling and guardrails, and wire in the memory ideas from here so it stays reliable over a long task.